In [1]:
import os
import pickle
from tqdm import tqdm
import torch
from token_utils_rep import EHRTokenizer
from dataset_utils_rep import HBERTPretrainEHRDataset, batcher
from torch.utils.data import DataLoader
from HEART_rep import HBERT_Pretrain
from set_seed_utils import set_random_seed

Disabling PyTorch because PyTorch >= 2.1 is required but found 1.13.1
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
args = {
    "seed": 0,
    "dataset": "MIMIC-IV",  # MIMIC-III, MIMIC-IV
    "batch_size": 32,
    "lr": 2e-5,
    "epochs": 20,
    "encoder": "hi",
    "mask_rate": 0.7,
    "anomaly_loss_weight": 1,
    "num_hidden_layers": 5,
    "num_attention_heads": 6,
    "attention_probs_dropout_prob": 0.2,
    "hidden_dropout_prob": 0.2,
    "edge_hidden_size": 32,
    "hidden_size": 288,  # must be divisible by num_attention_heads
    "intermediate_size": 288,
    "gnn_n_heads": 1,
    "gnn_temp": 1,
    "gat": "dotattn",  # dotattn, None
    "diag_med_emb": "tree",  # simple, tree
}

In [4]:
# here we only use the MIMIC dataset
args['max_visit_size'] = 15
args['predicted_token_type'] = ["diag"]
args['mask_token_id'] = {"diag":3}
args['special_tokens'] = ("[PAD]", "[CLS]", "[SEP]", 
                       "[MASK0]")
# note that here "[MASK0]", "[MASK1]", "[MASK2]", "[MASK3]" are used for masking pretraining task
# codes that are actually masked are not in the input sequence

In [5]:
full_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic.pkl"
pretrain_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic_pretrain.pkl" # for pretraining

In [6]:
ehr_data = pickle.load(open(full_data_path, 'rb'))

In [7]:
diag_sentences = ehr_data["ICD9_CODE"].values.tolist()
gender_set = [["M"], ["F"]]
age_set = [[c] for c in set(ehr_data["AGE"].values.tolist())]
age_gender_set = [[str(c) + "_" + gender] \
    for c in set(ehr_data["AGE"].values.tolist()) for gender in ["M", "F"]]

In [8]:
# tokenizer used full data
tokenizer = EHRTokenizer(diag_sentences, gender_set, 
                         age_set, age_gender_set, special_tokens=args["special_tokens"])

In [9]:
ehr_pretrain_data = pickle.load(open(pretrain_data_path, 'rb'))
pretrain_dataset = HBERTPretrainEHRDataset(ehr_pretrain_data, tokenizer, 
                                  token_type=args['predicted_token_type'], 
                                  mask_rate=args['mask_rate'])

In [10]:
pretrain_dataloader = DataLoader(pretrain_dataset, batch_size=args["batch_size"], 
                                 collate_fn=batcher(pad_id = tokenizer.vocab.word2id["[PAD]"], 
                                                    n_token_type=len(args["predicted_token_type"]), is_train = True),
                                 shuffle=True)

In [11]:
batch = next(iter(pretrain_dataloader))
input_ids, input_types, edge_index, visit_positions, labels = batch

In [12]:
set_random_seed(args["seed"])

[INFO] Random seed set to 0


In [13]:
exp_name = "Pretrain-HBERT" \
    + "-" + str(args["dataset"]) \
    + "-" + str(args["encoder"]) \
    + "-" + str(args["mask_rate"]) \
    + "-" + str(args["hidden_size"]) \
    + "-" + str(args["edge_hidden_size"]) \
    + "-" + str(args["num_hidden_layers"]) \
    + "-" + str(args["num_attention_heads"]) \
    + "-" + str(args["attention_probs_dropout_prob"]) \
    + "-" + str(args["hidden_dropout_prob"]) \
    + "-" + str(args["intermediate_size"]) \
    + "-" + str(args["gat"]) \
    + "-" + str(args["gnn_n_heads"]) \
    + "-" + str(args["gnn_temp"]) \
    + "-" + str(args["diag_med_emb"])
print(exp_name)

save_path = "./pretrained_models/" + exp_name
if not os.path.exists(save_path):
    os.makedirs(save_path)

Pretrain-HBERT-MIMIC-IV-hi-0.7-288-32-5-6-0.2-0.2-288-dotattn-1-1-tree


In [14]:
args["vocab_size"] = len(args["special_tokens"]) + len(tokenizer.diag_voc.id2word) + \
                len(tokenizer.age_voc.id2word) + \
                len(tokenizer.gender_voc.id2word) + \
                len(tokenizer.age_gender_voc.id2word)

args["label_vocab_size"] = {"diag":len(tokenizer.diag_voc.id2word)}  # {token_type: vocab_size}

In [15]:
loss_entity = ["diag"]

In [16]:
model = HBERT_Pretrain(args, tokenizer).to(device)

In [17]:
optimizer = torch.optim.AdamW(model.parameters(), lr=args["lr"])

In [18]:
for epoch in range(1, 1 + args["epochs"]):
    train_iter = tqdm(pretrain_dataloader, ncols=140)
    model.train()
    ave_loss, ave_loss_dict = 0., {token_type: 0. for token_type in loss_entity}

    for step, batch in enumerate(train_iter):

        batch = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
        loss, loss_dict, perf_dict = model(*batch)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        train_iter.set_description(f"Epoch:{epoch: 03d}, Step:{step: 03d}, loss:{loss.item():.4f}, diag:{loss_dict['diag']:.4f}")

        ave_loss += loss.item()
        ave_loss_dict = {token_type: ave_loss_dict[token_type] + loss_dict[token_type] for token_type in loss_entity}

    ave_loss /= (step + 1)
    ave_loss_dict = {token_type: ave_loss_dict[token_type] / (step + 1) for token_type in loss_entity}
    print(f"Epoch {epoch} finished, ave_loss: {ave_loss:.4f}, ave_loss_dict: {ave_loss_dict}, perf_dict: {perf_dict}")

  0%|                                                                                                              | 0/1328 [00:00<?, ?it/s]


AttributeError: 'HBERT_Pretrain' object has no attribute 'embeddings'

In [ ]:
torch.save(model.cpu().state_dict(), f"{save_path}/pretrained_model.pt")